# 🛡️ RMIT HACKATHON 2026 — UIT_SGS_Foodseekers_C1
### Multilingual Prompt-Attack Classification (BENIGN, JAILBREAK, PROMPT_INJECTION)
### Backbone: `FacebookAI/xlm-roberta-base` với Class-Weighted Loss & Dynamic Thresholding
---
**Quy cách Kaggle:** 100% Offline từ `/kaggle/input`, không gọi mạng ra ngoài, tự động nhận diện GPU T4.

In [ ]:
# 1. KIỂM TRA THIẾT BỊ GPU TRÊN KAGGLE
import os, gc, copy, random, numpy as np, pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from scipy.special import softmax
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"[*] Đã kích hoạt GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Không phát hiện GPU — Hãy bật T4 GPU trong mục Settings > Accelerator!")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. TỰ ĐỘNG ĐỌC DỮ LIỆU TỪ /kaggle/input (OFFLINE 100%)
DATA_DIR = "/kaggle/input/rmit-hackathon-2026"
if not os.path.exists(DATA_DIR):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            DATA_DIR = root
            break

print(f"[*] Đang nạp dữ liệu từ: {DATA_DIR}")
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
train_df["text"] = train_df["text"].fillna("").astype(str)
test_df["text"] = test_df["text"].fillna("").astype(str)

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for i, l in enumerate(LABELS)}
train_df["target"] = train_df["label"].map(L2I)

print(f"[+] Kích thước: Train {train_df.shape} | Test {test_df.shape}")
print(f"[+] Phân bố nhãn train: {train_df['label'].value_counts().to_dict()}")

# Trọng số lớp cân bằng chống thiên lệch nhãn hiếm 7% PROMPT_INJECTION
class_counts = train_df["target"].value_counts().sort_index().values
class_w = len(train_df) / (len(LABELS) * class_counts)
class_w_tensor = torch.tensor(class_w, dtype=torch.float).to(DEVICE)
print("[+] Trọng số phạt lỗi nhãn (Class Weights):", class_w_tensor.cpu().numpy().round(3))

In [ ]:
# 3. TÌM TRỌNG SỐ MÔ HÌNH OFFLINE TRÊN KAGGLE
MODEL_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(k in root.lower() for k in ["roberta", "xlm", "deberta", "model"]):
        MODEL_PATH = root
        break

if MODEL_PATH is None:
    # Mặc định dùng mã mô hình online nếu máy đã bật Internet
    MODEL_PATH = "FacebookAI/xlm-roberta-base"

print(f"[*] Sử dụng mô hình từ: {MODEL_PATH}")
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

class PromptDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], (-1 if self.labels is None else int(self.labels[idx]))

def collate_fn(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

# Chia tập Stratified theo cả Nhãn lẫn Ngôn ngữ để thẩm định chuẩn nhất
strat = train_df["label"] + "_" + train_df.get("language", pd.Series(["unk"]*len(train_df)))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(skf.split(train_df, strat))

tr_sub = train_df.iloc[train_idx].reset_index(drop=True)
va_sub = train_df.iloc[val_idx].reset_index(drop=True)

dl_tr = DataLoader(PromptDataset(tr_sub["text"], tr_sub["target"].values), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dl_va = DataLoader(PromptDataset(va_sub["text"], va_sub["target"].values), batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn)
dl_te = DataLoader(PromptDataset(test_df["text"]), batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn)
print(f"[+] Huấn luyện: {len(tr_sub)} dòng | Thẩm định: {len(va_sub)} dòng | Dự đoán: {len(test_df)} dòng")

In [ ]:
# 4. KHỞI TẠO MÔ HÌNH VÀ VÒNG LẶP HUẤN LUYỆN ĐO MACRO F1
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=3,
    id2label=I2L,
    label2id=L2I,
    ignore_mismatched_sizes=True
).to(DEVICE)

loss_fn = nn.CrossEntropyLoss(weight=class_w_tensor)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(dl_tr) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def evaluate_logits(dataloader):
    model.eval()
    all_logits = []
    for batch in dataloader:
        _ = batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(**batch)
        all_logits.append(out.logits.float().cpu().numpy())
    return np.concatenate(all_logits, axis=0)

best_val_f1 = -1.0
best_model_weights = None
y_va_true = va_sub["target"].values

for ep in range(EPOCHS):
    model.train()
    running_loss = 0.0
    pbar = tqdm(dl_tr, desc=f"Epoch {ep+1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(**batch).logits
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()
        pbar.set_postfix({"loss": f"{running_loss / (step + 1):.4f}"})
        
    val_logits = evaluate_logits(dl_va)
    val_f1 = f1_score(y_va_true, val_logits.argmax(axis=1), average="macro")
    print(f"\n[Epoch {ep+1}] Validation Macro F1 (Argmax thuần): {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_weights = copy.deepcopy(model.state_dict())
        print(f"[*] Đã lưu kỷ lục mô hình tốt nhất: {best_val_f1:.4f}!")

model.load_state_dict(best_model_weights)
print(f"\n🏆 ĐIỂM THẨM ĐỊNH TỐT NHẤT: {best_val_f1:.4f}")

In [ ]:
# 5. TINH CHỈNH TRỌNG SỐ NGƯỠNG CHO NHÃN HIẾM TRÊN TẬP VALIDATION
va_logits = evaluate_logits(dl_va)
va_probs = softmax(va_logits, axis=1)

best_tuned_f1 = best_val_f1
best_multipliers = (1.0, 1.0, 1.0)

for w_jb in np.arange(0.8, 1.35, 0.05):
    for w_pi in np.arange(0.9, 1.80, 0.05):
        multipliers = np.array([1.0, w_jb, w_pi])
        preds = (va_probs * multipliers).argmax(axis=1)
        score = f1_score(y_va_true, preds, average="macro")
        if score > best_tuned_f1:
            best_tuned_f1 = score
            best_multipliers = multipliers

print("=" * 65)
print(f"🏆 KẾT QUẢ SAU TINH CHỈNH NGƯỠNG: Macro F1 = {best_tuned_f1:.4f}")
print(f"   Hệ số tối ưu: BENIGN=1.0, JAILBREAK={best_multipliers[1]:.2f}, PROMPT_INJECTION={best_multipliers[2]:.2f}")
print("=" * 65)
final_va_preds = (va_probs * best_multipliers).argmax(axis=1)
print(classification_report(y_va_true, final_va_preds, target_names=LABELS, digits=4))

In [ ]:
# 6. DỰ ĐOÁN TẬP TEST VÀ XUẤT SUBMISSION.CSV CHUẨN QUY CÁCH
print("[*] Đang dự đoán trên toàn bộ 6.000 mẫu tập test...")
test_logits = evaluate_logits(dl_te)
test_probs = softmax(test_logits, axis=1)
final_test_preds = (test_probs * best_multipliers).argmax(axis=1)

id_col = "id" if "id" in test_df.columns else "Id"
sub = pd.DataFrame({
    "id": test_df[id_col],
    "label": [I2L[i] for i in final_test_preds]
})

# Kiểm tra toàn vẹn bắt buộc của cuộc thi
assert len(sub) == 6000, f"Số dòng không khớp 6000: {len(sub)}"
assert sub["label"].isin(LABELS).all(), "Có nhãn không hợp lệ!"
assert sub["id"].is_unique, "Trùng ID!"

sub.to_csv("submission.csv", index=False)
print("[✅] ĐÃ XUẤT THÀNH CÔNG FILE submission.csv ĐẠT CHUẨN 100%!")
print("[i] Phân phối dự đoán tập test:")
print(sub["label"].value_counts().to_dict())
print(sub.head(10))